In [ ]:
!pip install -r ../requirements.txt

## Qdrant란?

Qdrant는 고차원 임베딩 벡터를 저장하고, 근사 최근접 탐색(ANN)을 통해 의미적으로 유사한 데이터를 빠르게 검색할 수 있도록 설계된 오픈소스 벡터 데이터베이스다.
생성형 AI와 RAG 아키텍처가 확산되면서, 검색 단계의 정확도와 응답 품질이 서비스 성능을 좌우하게 되었고 Qdrant의 활용도도 함께 높아졌다.
Qdrant는 단순 벡터 유사도 검색을 넘어, 메타데이터 기반 필터링과 하이브리드 검색을 지원해 실제 서비스 환경에 적합한 검색 구조를 제공한다.
또한 실시간 업데이트, 다양한 인덱싱 전략, 운영 편의성을 갖춰 챗봇·문서 검색·추천 시스템 등에서 폭넓게 활용된다.

In [16]:
# Cell 2: OpenAI, dotenv, Qdrant client 초기화
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

DATA_DIR = Path("../data")
QDRANT_PATH = DATA_DIR / "qdrant_store"
COLLECTION_NAME = "demo_rag_docs"
EMBEDDING_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-mini"

qdrant = QdrantClient(path=str(QDRANT_PATH))

print("데이터 폴더:", DATA_DIR.resolve())
print("Qdrant 로컬 저장 경로:", QDRANT_PATH.resolve())

RuntimeError: Storage folder ../data/qdrant_store is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

OpenAI API를 쓰기 사용하기 위해 클라이언트를 선언하는 것처럼, Qdrant를 사용하기 위해서도 클라이언트를 선언해야한다.

QdrantClient(url="..."): 원격 Qdrant 서버에 접속
QdrantClient(paht="..."): 내 컴퓨터 로컬에 저장하여 사용

Embedding 모델은 'text-embedding-3-small'을 사용한다.

In [ ]:
# Cell 3: 샘플 문서 정의
doc_paths = [
    DATA_DIR / "company_intro.txt",
    DATA_DIR / "refund_policy.txt",
    DATA_DIR / "business_hours.txt",
    DATA_DIR / "contact_info.txt",
]

documents = []

for doc_id, path in enumerate(doc_paths, start=1):
    text = path.read_text(encoding="utf-8").strip()
    title = text.splitlines()[0]

    documents.append(
        {
            "id": doc_id,
            "title": title,
            "category": path.stem,
            "source": path.name,
            "text": text,
        }
    )

for doc in documents:
    preview = doc["text"][:80].replace("\n", " ")
    print(f'{doc["id"]}. {doc["title"]} | {doc["source"]}')
    print(preview)
    print()

1. 회사 소개 | company_intro.txt
회사 소개  에이아이 랩 스토어는 직장인과 대학생을 위한 AI 학습 콘텐츠를 제공하는 온라인 교육 플랫폼입니다. 주요 서비스는 실습형 강의, 주

2. 환불 정책 | refund_policy.txt
환불 정책  결제 후 7일 이내이며 강의를 한 번도 수강하지 않은 경우 전액 환불이 가능합니다. 결제 후 7일이 지났거나 강의 영상을 20퍼센트

3. 운영 시간 | business_hours.txt
운영 시간  고객센터 운영 시간은 평일 오전 10시부터 오후 6시까지입니다. 점심시간은 오후 12시 30분부터 오후 1시 30분까지이며 해당 시

4. 연락처 | contact_info.txt
연락처  대표 이메일은 support@ailabstore.example 입니다. 대표 전화번호는 02-1234-5678 입니다. 카카오톡 상담 



In [ ]:
# Cell 4: 임베딩 함수 만들기
def get_embedding(text: str) -> list[float]:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text,
    )
    return response.data[0].embedding

sample_vector = get_embedding(documents[0]["text"])
print("임베딩 차원:", len(sample_vector))

임베딩 차원: 1536


openAI API를 사용하여 text를 벡터로 변환한다.

In [ ]:
# Cell 5: Qdrant collection 생성
from qdrant_client.models import Distance, VectorParams

try:
    qdrant.delete_collection(COLLECTION_NAME)
except Exception:
    pass

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    # 벡터 스키마 정의
    vectors_config=VectorParams(
        size=len(sample_vector),
        distance=Distance.COSINE,
    ),
)

print(f"'{COLLECTION_NAME}' 컬렉션 생성 완료")

'demo_rag_docs' 컬렉션 생성 완료


벡터 db는 데이터를 저장하기 위해서 `Collection`이라는 논리적 단위를 사용한다.
`Collection`을 RDB에 비유하면 Table에 가깝다.

In [ ]:
# Cell 6: 문서 upsert
from qdrant_client.models import PointStruct

points = []

for doc in documents:
    vector = get_embedding(doc["text"])
    points.append(
        PointStruct(
            id=doc["id"],
            vector=vector,
            payload={
                "title": doc["title"],
                "category": doc["category"],
                "source": doc["source"],
                "text": doc["text"],
            },
        )
    )

qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

print(f"{len(points)}개의 문서를 Qdrant에 저장했습니다.")

4개의 문서를 Qdrant에 저장했습니다.


`PointStruct`는 Qdrant에 넣는 개별 데이터 1건의 구조체이다.
쉽게 말하면 컬렉션 안에 저장되는 한 행을 표현하는 객체다.

id = pk와 비슷한 값

vector = 검색용 임베딩

payload = 부가 정보 컬럼들

>payload는 검색 대상 벡터에 붙는 메타데이터이자 원문 정보로, 필터링·출처관리·LLM 전달을 위해 필요하다.

In [ ]:
# Cell 7: 검색 함수 만들기
def search_documents(query: str, limit: int = 3):
    query_vector = get_embedding(query)
    response = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=limit,
        with_payload=True,
    )
    return response.points

사용자 질문 또한 벡터화를 진행한다.  
사용자 질문에 대해 유사도가 높은 단어들을 가져온다.  
Qdrant에서 벡터 쿼리를 생성할 때는 `query_points` 메서드를 사용한다.

In [ ]:
# Cell 8: 검색 결과 출력
query = "환불은 언제까지 가능하고 어디로 문의하면 돼?"
search_results = search_documents(query, limit=3)

print("질문:", query)
print()

for idx, result in enumerate(search_results, start=1):
    payload = result.payload
    print(f"[{idx}] score={result.score:.4f}")
    print("title:", payload["title"])
    print("source:", payload["source"])
    print(payload["text"])
    print()

질문: 환불은 언제까지 가능하고 어디로 문의하면 돼?

[1] score=0.5140
title: 환불 정책
source: refund_policy.txt
환불 정책

결제 후 7일 이내이며 강의를 한 번도 수강하지 않은 경우 전액 환불이 가능합니다.
결제 후 7일이 지났거나 강의 영상을 20퍼센트 이상 수강한 경우에는 전액 환불이 어렵습니다.
강의 영상을 20퍼센트 미만으로 수강한 경우에는 수강한 분량을 제외한 금액 기준으로 부분 환불이 가능합니다.
라이브 세션이 포함된 클래스는 라이브 세션 시작 24시간 전까지만 환불 요청이 가능합니다.
환불은 고객센터 이메일 또는 대표 전화로 접수할 수 있으며 접수 후 영업일 기준 3일 이내 처리됩니다.
이벤트, 쿠폰, 프로모션으로 구매한 상품은 별도 환불 조건이 적용될 수 있습니다.

[2] score=0.3341
title: 연락처
source: contact_info.txt
연락처

대표 이메일은 support@ailabstore.example 입니다.
대표 전화번호는 02-1234-5678 입니다.
카카오톡 상담 채널 이름은 에이아이 랩 스토어 고객센터입니다.
환불, 결제, 수강 관련 문의는 이메일로 남기면 가장 빠르게 처리됩니다.
제휴 및 기업 교육 문의는 biz@ailabstore.example 로 보낼 수 있습니다.

[3] score=0.2843
title: 운영 시간
source: business_hours.txt
운영 시간

고객센터 운영 시간은 평일 오전 10시부터 오후 6시까지입니다.
점심시간은 오후 12시 30분부터 오후 1시 30분까지이며 해당 시간에는 전화 상담이 어렵습니다.
토요일, 일요일, 공휴일에는 고객센터를 운영하지 않습니다.
온라인 강의 시청과 자료 다운로드는 24시간 이용할 수 있습니다.
라이브 세션 문의는 세션 시작 1시간 전까지 접수하는 것을 권장합니다.



결과로 score와 이전에 저장해두었던 payload 정보도 확인 할 수 있다.

In [ ]:
# Cell 9: RAG 답변 생성
def generate_rag_answer(query: str, limit: int = 3):
    results = search_documents(query, limit=limit)
    context_blocks = []

    for idx, result in enumerate(results, start=1):
        payload = result.payload
        context_blocks.append(
            f"[문서 {idx}] {payload['title']} ({payload['source']})\n{payload['text']}"
        )

    context = "\n\n".join(context_blocks)

    response = client.responses.create(
        model=CHAT_MODEL,
        instructions=(
            "너는 사내 문서 검색을 도와주는 AI 어시스턴트다. "
            "반드시 제공된 문맥 안에서만 답하고, 문맥에 없으면 모른다고 말해라. "
            "답변 마지막에는 참고한 문서 제목을 짧게 적어라."
        ),
        input=f"사용자 질문:\n{query}\n\n참고 문서:\n{context}",
    )

    return response.output_text, results

rag_answer, rag_results = generate_rag_answer(query)
print(rag_answer)

환불은 결제 후 7일 이내이며 강의를 한 번도 수강하지 않은 경우 전액 환불이 가능하고, 라이브 세션이 포함된 클래스는 라이브 세션 시작 24시간 전까지만 환불 요청이 가능합니다. 환불 문의는 고객센터 이메일(support@ailabstore.example) 또는 대표 전화(02-1234-5678)로 할 수 있으며, 이메일 문의가 가장 빠르게 처리됩니다.

참고: 환불 정책, 연락처


벡터 검색을 통해 생성된 결과를 쿼리에 포함시켜 LLM이 문서 기반 응답을 생성하도록 유도한다.

In [17]:
# Cell 10: 일반 답변 vs RAG 답변 비교
plain_response = client.responses.create(
    model=CHAT_MODEL,
    instructions="너는 친절한 고객센터 AI 어시스턴트다.",
    input=query,
)

print("[일반 답변]")
print(plain_response.output_text)
print()
print("[RAG 답변]")
print(rag_answer)

[일반 답변]
안녕하세요! 환불 가능 기간과 문의 방법에 대해 안내드리겠습니다.

1. 환불 가능 기간: 보통 구매일로부터 7일 이내이지만, 상품이나 서비스에 따라 다를 수 있으니 구체적인 구매 내역을 확인해 주세요.

2. 문의 방법: 
- 고객센터 전화번호: 1234-5678
- 이메일: support@example.com
- 홈페이지 내 1:1 문의 게시판

구체적인 환불 조건이나 절차가 필요하시면 구매하신 곳의 환불 정책을 참고해 주시거나, 위 연락처로 문의해 주시면 상세히 안내해 드리겠습니다. 도움이 필요하시면 언제든 말씀해 주세요!

[RAG 답변]
환불은 결제 후 7일 이내이며 강의를 한 번도 수강하지 않은 경우 전액 환불이 가능하고, 라이브 세션이 포함된 클래스는 라이브 세션 시작 24시간 전까지만 환불 요청이 가능합니다. 환불 문의는 고객센터 이메일(support@ailabstore.example) 또는 대표 전화(02-1234-5678)로 할 수 있으며, 이메일 문의가 가장 빠르게 처리됩니다.

참고: 환불 정책, 연락처


RAG를 사용하지 않은 응답과 비교했을 때, 훨씬 더 정확한 응답을 내뱉는 것을 확인 할 수 있다.

## 오늘 배운 점

1. Qdrant는 collection이란 단위로 컬럼들을 관리한다.
2. RAG를 사용한 응답 생성 순서는 문서 임베딩->벡터 db 삽입->쿼리 임베딩->벡터 db 검색->검색 결과와 쿼리를 llm에게 전송->문서를 기반으로 응답 생성 순으로 이뤄진다.
3. 